## Example 1: Basic priority queue operations

In [ ]:
from heapdict import heapdict

# Example 1: Basic priority queue operations
print("Example 1: Basic Priority Queue Operations")

# Create heap dictionary
hd = heapdict()
print(f"Created heapdict: {type(hd).__name__}")

# Add items with priorities
tasks = [
    ("send_email", 3),
    ("backup_data", 1),
    ("update_system", 2),
    ("user_meeting", 5),
    ("bug_fix", 4)
]

print(f"Adding {len(tasks)} tasks with priorities:")
for task, priority in tasks:
    hd[task] = priority
    print(f"  {task}: priority {priority}")

assert len(hd) == 5

# Pop items in priority order (lowest first)
print(f"\nProcessing tasks by priority:")
order = []
while hd:
    task, priority = hd.popitem()
    order.append(task)
    print(f"  Processing: {task} (priority {priority})")

assert order == ["backup_data", "update_system", "send_email", "bug_fix", "user_meeting"], order
assert len(hd) == 0

# Demonstrate key operations
hd2 = heapdict()
hd2['urgent'] = 1
hd2['normal'] = 5
hd2['low'] = 10

print(f"\nKey operations:")
print(f"  Peek minimum: {hd2.peekitem()}")
print(f"  Contains 'urgent': {'urgent' in hd2}")
print(f"  Get priority: hd2['normal'] = {hd2['normal']}")

# peekitem returns the minimum without removing it
assert hd2.peekitem() == ("urgent", 1) and len(hd2) == 3
assert "urgent" in hd2 and hd2["normal"] == 5

## Example 2: Priority updates

In [ ]:
from heapdict import heapdict

# Example 2: Priority updates and dynamic modifications
print("Example 2: Priority Updates and Dynamic Modifications")

# Create process queue with initial priorities
processes = heapdict()
processes['database_backup'] = 10
processes['user_request'] = 5
processes['system_monitor'] = 8
processes['log_cleanup'] = 15

print("Initial process queue:")
for process, priority in sorted(processes.items(), key=lambda x: x[1]):
    print(f"  {process}: {priority}")

print(f"\nNext process to run: {processes.peekitem()}")
assert processes.peekitem() == ("user_request", 5)

# Update priority of existing process
print("\nUrgent: database_backup priority increased!")
processes['database_backup'] = 2  # Higher priority (lower number)

# Add new urgent process
processes['security_alert'] = 1
print("Added security_alert with highest priority")

print(f"\nUpdated queue - next process: {processes.peekitem()}")

# Process items in priority order
assert processes.peekitem() == ("security_alert", 1)

print("\nProcessing queue:")
executed = []
while processes:
    process, priority = processes.popitem()
    executed.append((process, priority))
    print(f"  Executing: {process} (priority {priority})")

# the lowered priority moved database_backup ahead of user_request
assert executed == [
    ("security_alert", 1),
    ("database_backup", 2),
    ("user_request", 5),
    ("system_monitor", 8),
    ("log_cleanup", 15),
], executed

## Example 3: Task scheduling by deadline

In [ ]:
from heapdict import heapdict
import time

# Example 3: Task scheduling simulation
print("Example 3: Task Scheduling Simulation")

# Simulate task scheduler with deadlines
scheduler = heapdict()

# Add tasks with deadline timestamps (lower = more urgent)
current_time = time.time()
tasks = [
    ("send_report", current_time + 3600),     # Due in 1 hour
    ("team_meeting", current_time + 1800),   # Due in 30 minutes  
    ("code_review", current_time + 7200),    # Due in 2 hours
    ("daily_standup", current_time + 900),   # Due in 15 minutes
    ("lunch_break", current_time + 5400)     # Due in 1.5 hours
]

print("Adding tasks to scheduler:")
for task, deadline in tasks:
    scheduler[task] = deadline
    remaining = int(deadline - current_time)
    print(f"  {task}: due in {remaining//60} minutes")

# Process tasks by deadline
print(f"\nProcessing {len(scheduler)} tasks by urgency:")
processed = []
while scheduler:
    task, deadline = scheduler.popitem()
    processed.append(task)
    remaining = int(deadline - current_time)
    print(f"  Next: {task} (due in {remaining//60} minutes)")
    
    # Simulate task processing
    if remaining < 1200:  # Less than 20 minutes
        print(f"    ⚠️  Urgent task!")
    
# earliest deadline first
assert processed == ["daily_standup", "team_meeting", "send_report", "lunch_break", "code_review"], processed

## Example 4: Dictionary interface (lookup, update, delete)

In [ ]:
from heapdict import heapdict

# Example 4: Heap operations and dictionary interface
print("Example 4: Heap Operations and Dictionary Interface")

# Create and populate heapdict
server_loads = heapdict()
servers = ["web-1", "web-2", "web-3", "api-1", "api-2"]
loads = [85, 42, 67, 15, 91]

# Add server load data
for server, load in zip(servers, loads):
    server_loads[server] = load
    
print("Server load balancing:")
print(f"Total servers: {len(server_loads)}")

# Check specific server
print(f"web-1 load: {server_loads['web-1']}%")

# Find least loaded server
least_loaded = server_loads.peekitem()
print(f"Least loaded server: {least_loaded[0]} ({least_loaded[1]}%)")
assert least_loaded == ("api-1", 15)

# Update server load
server_loads['api-1'] = 85  # Load increased
print(f"Updated api-1 load to {server_loads['api-1']}%")

# Show current minimum after update
new_min = server_loads.peekitem()
print(f"New least loaded: {new_min[0]} ({new_min[1]}%)")
assert new_min == ("web-2", 42)  # the heap re-orders after a priority change

# Test membership and deletion
print(f"'web-3' in servers: {'web-3' in server_loads}")
del server_loads['web-3']
print(f"Removed web-3, remaining servers: {len(server_loads)}")
assert "web-3" not in server_loads and len(server_loads) == 4

# Drain the two least loaded servers
print("Processing servers by load:")
drained = [server_loads.popitem() for _ in range(2)]
for server, load in drained:
    print(f"  {server}: {load}%")
assert drained[0] == ("web-2", 42) and drained[1][1] == 85

## Example 5: Shortest path with heapdict as the priority queue

In [ ]:
from heapdict import heapdict

# Example 5: Simple shortest path algorithm (Dijkstra-like)
print("Example 5: Simple Shortest Path Algorithm")

# Simple graph represented as adjacency list with weights
graph = {
    'A': [('B', 4), ('C', 2)],
    'B': [('C', 1), ('D', 5)], 
    'C': [('D', 8), ('E', 10)],
    'D': [('E', 2)],
    'E': []
}

def shortest_path(graph, start, end):
    # Use heapdict for priority queue of distances
    distances = heapdict()
    distances[start] = 0
    
    # Track shortest distances found
    visited = {}
    
    while distances:
        current, dist = distances.popitem()
        
        if current in visited:
            continue
            
        visited[current] = dist
        
        if current == end:
            return dist
            
        # Update distances to neighbors
        for neighbor, weight in graph.get(current, []):
            if neighbor not in visited:
                new_dist = dist + weight
                if neighbor not in distances or new_dist < distances[neighbor]:
                    distances[neighbor] = new_dist
    
    return float('inf')

# Find shortest path from A to E
distance = shortest_path(graph, 'A', 'E')
print(f"Shortest path from A to E: {distance}")
assert distance == 11  # A -> B -> D -> E (4 + 5 + 2)
assert shortest_path(graph, 'A', 'C') == 2
assert shortest_path(graph, 'E', 'A') == float('inf')